# Re-ID experiment v2 — OSNet\nRuntime -> Change runtime type -> **T4 GPU** -> Run all. Upload the same `reid_experiment.zip` at Cell 1.

In [ ]:
# Cell 1 — upload reid_experiment.zip when prompted
from google.colab import files
up = files.upload()
import zipfile, os
zipfile.ZipFile(list(up)[0]).extractall("data")
print(len(os.listdir("data/crops")), "crops")

In [ ]:
# Cell 2 — OSNet re-ID embeddings
!pip -q install git+https://github.com/KaiyangZhou/deep-person-reid.git
import torchreid, torch, glob, numpy as np
from PIL import Image
import torchvision.transforms as T
dev = "cuda" if torch.cuda.is_available() else "cpu"
m = torchreid.models.build_model("osnet_x1_0", num_classes=1, pretrained=True).to(dev).eval()
tf = T.Compose([T.Resize((256,128)), T.ToTensor(),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
files_ = sorted(glob.glob("data/crops/*.jpg"))
embs = []
with torch.no_grad():
    for i in range(0, len(files_), 64):
        b = torch.stack([tf(Image.open(f).convert("RGB")) for f in files_[i:i+64]]).to(dev)
        embs.append(m(b).cpu().numpy())
X = np.concatenate(embs); X = X/np.linalg.norm(X,axis=1,keepdims=True)
np.save("emb_osnet.npy", X)
print("OSNet embeddings:", X.shape)

In [ ]:
# Cell 3 — cluster + montages at several thresholds
import cv2, numpy as np, glob
from sklearn.cluster import AgglomerativeClustering
X = np.load("emb_osnet.npy"); files_ = sorted(glob.glob("data/crops/*.jpg"))
for THR in (0.25, 0.35, 0.45):
    lab = AgglomerativeClustering(n_clusters=None, distance_threshold=THR,
          metric="cosine", linkage="average").fit(X).labels_
    sizes = np.bincount(lab)
    big = [c for c in np.argsort(-sizes) if sizes[c] >= 8]
    print(f"THR={THR}: {lab.max()+1} clusters, {len(big)} with >=8 crops, sizes {sorted(sizes,reverse=True)[:12]}")
    rows = []
    for c in big[:22]:
        idx = [i for i in range(len(files_)) if lab[i]==c]
        idx = idx[::max(1,len(idx)//10)][:10]
        tiles = []
        for i in idx:
            im = cv2.imread(files_[i]); s = 110/im.shape[0]
            tiles.append(cv2.resize(im,(max(16,int(im.shape[1]*s)),110)))
        w = sum(t.shape[1] for t in tiles)+len(tiles)*3+60
        cv = np.zeros((116,w,3),np.uint8)
        cv2.putText(cv,f"C{c}:{sizes[c]}",(2,60),0,0.5,(255,255,255),1)
        x = 60
        for t in tiles: cv[3:113,x:x+t.shape[1]]=t; x+=t.shape[1]+3
        rows.append(cv)
    wm = max(r.shape[1] for r in rows)
    rows = [np.pad(r,((0,0),(0,wm-r.shape[1]),(0,0))) for r in rows]
    cv2.imwrite(f"clusters_osnet_thr{int(THR*100)}.jpg", np.vstack(rows))
print("montages written")

In [ ]:
# Cell 4 — download results
import zipfile, glob
with zipfile.ZipFile("results_osnet.zip","w") as z:
    for f in glob.glob("clusters_osnet_*.jpg"): z.write(f)
    z.write("emb_osnet.npy")
from google.colab import files
files.download("results_osnet.zip")